# Notebook 03: ST-GCN v3.1 Action Recognition — Systematic Ablation & Training
## Two-Stage Human Action Recognition System (Fall Detection)

This notebook develops and evaluates **ST-GCN v3.1 (Proposed Model)** on **Dataset V3b** via a controlled incremental ablation study:
- **Baseline Reference**: `ST-GCN v3.0-clean` (`v2.4` architecture trained on group-heldout Dataset V3b), `ST-GCN v2.4` (`action_stgcn_v2_4.pth`), and `LSTM` (`action_lstm.pth`) are preserved intact.
- **Incremental Ablation Ladder (Evaluated Strictly on Validation Set: `830` windows / `143` Val clips)**:
  1. **`ST-GCN v3.0-clean (Baseline)`**: `v2.4` architecture (3-Subset Spatial Partitioning + Adaptive Graph `A_adap` + Multi-Scale TCN + Temporal Attention Pyramid + 9-Channel Joint/Bone/Motion Kinematics).
  2. **`ST-GCN v3.1-A (+ Gravity-Aware Features)`**: `v3.0-clean` + signed vertical acceleration, height above virtual ground anchor, spine orientation $(\cos\theta, |\sin\theta|)$, skeleton aspect ratio $\text{AR}$, and virtual hip vertical trajectory.
  3. **`ST-GCN v3.1-B (+ Wrist-to-Head Features)`**: `v3.0-clean` + wrist-to-nose relative vectors, Euclidean distances $(d_L, d_R, d_{\min})$, elbow flexion cosines, and temporal oscillation descriptors.
  4. **`ST-GCN v3.1-C (+ Lower-Body Kinematics)`**: `v3.0-clean` + knee flexion cosines (`hip-knee-ankle`), vertical segment drops (`hip-knee`, `knee-ankle`, `hip-ankle`), and foot stance width.
  5. **`ST-GCN v3.1-D1 / D2 / D3 (+ Feature Fusion)`**: Multi-branch combinations (`A+B`, `A+C`, and full `A+B+C`).
- **Scientific Discipline**: Architecture and loss-weighting selection are performed **100% on the Validation set (`830` windows / `143` Val clips)**. The held-out Test set (`906` windows / `147` Test clips: `V1_LEGACY_TEST` + `V2_GROUP_HELDOUT_TEST`) is evaluated **only once** at the end on the winning Validation checkpoint.

## 1. Environment & Reproducibility Setup

In [ ]:
import collections
import json
import os
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. ST-GCN v3.1 Architecture (`v3.0-clean` → `v3.1-A` → `v3.1-B` → `v3.1-C` → `v3.1-D` Configurable)

In [ ]:
NUM_JOINTS = 17
NUM_COORDS = 3
COCO_PARENTS = [0, 0, 0, 1, 2, 0, 0, 5, 6, 7, 8, 5, 6, 11, 12, 13, 14]

def build_spatial_partitions(num_joints: int = NUM_JOINTS) -> torch.Tensor:
    """
    3-subset spatial partition adjacency tensor A of shape (3, V, V):
      • A[0]: Self-connections (Identity I)
      • A[1]: Inward (centripetal) edges toward torso core
      • A[2]: Outward (centrifugal) edges away from torso core
    """
    inward_edges = [
        (1, 0), (2, 0), (3, 1), (4, 2),              # face -> nose
        (0, 5), (0, 6),                              # nose -> shoulders
        (7, 5), (9, 7), (8, 6), (10, 8),             # arms -> shoulders
        (5, 11), (6, 12), (5, 6), (11, 12),          # torso ring
        (13, 11), (15, 13), (14, 12), (16, 14),      # legs -> hips
    ]
    A_self = np.eye(num_joints, dtype=np.float32)
    A_in   = np.zeros((num_joints, num_joints), dtype=np.float32)
    A_out  = np.zeros((num_joints, num_joints), dtype=np.float32)
    for child, parent in inward_edges:
        if child < num_joints and parent < num_joints:
            A_in[child, parent] = 1.0
            A_out[parent, child] = 1.0

    def _norm_digraph(M: np.ndarray) -> np.ndarray:
        deg = M.sum(axis=1, keepdims=True)
        return np.where(deg > 0, M / np.maximum(deg, 1e-6), 0.0).astype(np.float32)

    return torch.from_numpy(np.stack([A_self, _norm_digraph(A_in), _norm_digraph(A_out)], axis=0))

class STGCNv2Block(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        A: torch.Tensor,
        stride: int = 1,
        use_adaptive_adj: bool = True,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.K = int(A.shape[0])
        self.register_buffer('A', A)
        self.use_adaptive_adj = use_adaptive_adj
        self.edge_importance = nn.Parameter(torch.ones_like(A))
        if use_adaptive_adj:
            self.A_adap = nn.Parameter(torch.zeros_like(A))
        else:
            self.register_parameter('A_adap', None)

        self.gcn = nn.Conv2d(in_channels, out_channels * self.K, kernel_size=1)
        self.bn_s = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

        mid_c = out_channels // 2
        self.tcn_b1 = nn.Conv2d(out_channels, mid_c, kernel_size=(3, 1), padding=(1, 0), stride=(stride, 1))
        self.tcn_b2 = nn.Conv2d(out_channels, out_channels - mid_c, kernel_size=(5, 1), padding=(2, 0), stride=(stride, 1))
        self.bn_t = nn.BatchNorm2d(out_channels)
        self.drop = nn.Dropout(dropout)

        self.residual = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
            if in_channels != out_channels or stride != 1
            else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, T, V = x.shape
        A_eff = self.A * self.edge_importance
        if self.use_adaptive_adj and self.A_adap is not None:
            A_eff = A_eff + self.A_adap

        z = self.gcn(x).view(B, self.K, -1, T, V)
        y = torch.einsum('bkctv,kvw->bctw', z, A_eff)
        y = self.act(self.bn_s(y))

        y_t = torch.cat([self.tcn_b1(y), self.tcn_b2(y)], dim=1)
        y_t = self.drop(self.bn_t(y_t))
        return self.act(y_t + self.residual(x))

class ActionSTGCN(nn.Module):
    """
    Modular ST-GCN v2 supporting all 4 incremental ablation stages:
      • v2.1: use_temporal_pyr=False, use_adaptive_adj=False, use_kinematic_feats=False
      • v2.2: use_temporal_pyr=True,  use_adaptive_adj=False, use_kinematic_feats=False
      • v2.3: use_temporal_pyr=True,  use_adaptive_adj=True,  use_kinematic_feats=False
      • v2.4 / v3.0-clean: use_temporal_pyr=True, use_adaptive_adj=True, use_kinematic_feats=True
      • v3.1-A: + use_gravity_feats=True
      • v3.1-B: + use_wrist_head_feats=True
      • v3.1-C: + use_lower_body_feats=True
      • v3.1-D: + Feature Fusion combinations of (A, B, C)
    """
    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 8,
        num_joints: int = 17,
        use_temporal_pyr: bool = True,
        use_adaptive_adj: bool = True,
        use_kinematic_feats: bool = True,
        use_gravity_feats: bool = False,
        use_wrist_head_feats: bool = False,
        use_lower_body_feats: bool = False,
    ):
        super().__init__()
        self.use_temporal_pyr     = use_temporal_pyr
        self.use_adaptive_adj     = use_adaptive_adj
        self.use_kinematic_feats  = use_kinematic_feats
        self.use_gravity_feats    = use_gravity_feats
        self.use_wrist_head_feats = use_wrist_head_feats
        self.use_lower_body_feats = use_lower_body_feats
        self.register_buffer('parents', torch.tensor(COCO_PARENTS[:num_joints], dtype=torch.long))

        eff_in = in_channels * 3 if use_kinematic_feats else in_channels
        if use_gravity_feats:    eff_in += 2
        if use_wrist_head_feats: eff_in += 2
        if use_lower_body_feats: eff_in += 2
        A = build_spatial_partitions(num_joints)

        self.data_bn = nn.BatchNorm1d(eff_in * num_joints)
        self.b1 = STGCNv2Block(eff_in, 64,  A, stride=1, use_adaptive_adj=use_adaptive_adj, dropout=0.10)
        self.b2 = STGCNv2Block(64,     64,  A, stride=1, use_adaptive_adj=use_adaptive_adj, dropout=0.10)
        self.b3 = STGCNv2Block(64,     128, A, stride=2, use_adaptive_adj=use_adaptive_adj, dropout=0.20)
        self.b4 = STGCNv2Block(128,    128, A, stride=1, use_adaptive_adj=use_adaptive_adj, dropout=0.20)

        if self.use_temporal_pyr:
            self.temp_att = nn.Sequential(
                nn.Linear(128, 32),
                nn.Tanh(),
                nn.Linear(32, 1),
            )
            head_in = 128 * 3
        else:
            head_in = 128

        if self.use_gravity_feats:
            self.grav_mlp = nn.Sequential(nn.LayerNorm(24), nn.Linear(24, 32), nn.SiLU())
            head_in += 32
        if self.use_wrist_head_feats:
            self.wrist_mlp = nn.Sequential(nn.LayerNorm(24), nn.Linear(24, 32), nn.SiLU())
            head_in += 32
        if self.use_lower_body_feats:
            self.lower_mlp = nn.Sequential(nn.LayerNorm(24), nn.Linear(24, 32), nn.SiLU())
            head_in += 32

        self.head = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Dropout(0.3),
            nn.Linear(head_in, num_classes),
        )

    @staticmethod
    def _summarize_temporal_signals(sig: torch.Tensor) -> torch.Tensor:
        mid = max(1, sig.shape[2] // 2)
        s_mean = sig.mean(dim=2)
        s_std  = torch.sqrt(sig.var(dim=2, unbiased=False) + 1e-6)
        s_diff = sig[:, :, mid:].mean(dim=2) - sig[:, :, :mid].mean(dim=2)
        s_osc  = (sig[:, :, 1:] - sig[:, :, :-1]).abs().mean(dim=2) if sig.shape[2] > 1 else torch.zeros_like(s_mean)
        return torch.cat([s_mean, s_std, s_diff, s_osc], dim=1)

    @staticmethod
    def _joint_angle_cos(a_xy: torch.Tensor, b_xy: torch.Tensor, c_xy: torch.Tensor) -> torch.Tensor:
        u = a_xy - b_xy
        v = c_xy - b_xy
        dot = (u * v).sum(dim=1, keepdim=True)
        norm = torch.sqrt((u ** 2).sum(dim=1, keepdim=True) + 1e-6) * torch.sqrt((v ** 2).sum(dim=1, keepdim=True) + 1e-6)
        return torch.clamp(dot / (norm + 1e-6), -1.0, 1.0)

    def _compute_v31_branches(self, raw_x: torch.Tensor):
        x_c = raw_x[:, 0, :, :]
        y_c = raw_x[:, 1, :, :]
        vis = (raw_x[:, 2, :, :] >= 0.20).float()
        node_extras, head_extras = [], []
        low_y = y_c[:, :, 13:17]
        low_vis = vis[:, :, 13:17]
        any_low = (low_vis.sum(dim=-1, keepdim=True) > 0).float()
        ground_y = torch.where(low_vis > 0, low_y, torch.full_like(low_y, 10.0)).amin(dim=-1, keepdim=True) * any_low

        if self.use_gravity_feats:
            vy = torch.zeros_like(y_c)
            vis_t1 = vis[:, 1:, :] * vis[:, :-1, :]
            vy[:, 1:, :] = (y_c[:, 1:, :] - y_c[:, :-1, :]) * vis_t1
            ay = torch.zeros_like(y_c)
            vis_t2 = vis_t1[:, 1:, :] * vis_t1[:, :-1, :]
            ay[:, 2:, :] = (vy[:, 2:, :] - vy[:, 1:-1, :]) * vis_t2
            rel_ground = (y_c - ground_y) * vis
            node_extras.append(torch.stack([ay, rel_ground], dim=1))

            sh_x = 0.5 * (x_c[:, :, 5] + x_c[:, :, 6])
            sh_y = 0.5 * (y_c[:, :, 5] + y_c[:, :, 6])
            sh_norm = torch.sqrt(sh_x ** 2 + sh_y ** 2 + 1e-6)
            cos_theta = sh_y / sh_norm
            abs_sin_theta = sh_x.abs() / sh_norm
            any_vis = (vis.sum(dim=-1, keepdim=True) > 0).float()
            y_max = torch.where(vis > 0, y_c, torch.full_like(y_c, -10.0)).amax(dim=-1, keepdim=True) * any_vis
            y_min = torch.where(vis > 0, y_c, torch.full_like(y_c, +10.0)).amin(dim=-1, keepdim=True) * any_vis
            x_max = torch.where(vis > 0, x_c, torch.full_like(x_c, -10.0)).amax(dim=-1, keepdim=True) * any_vis
            x_min = torch.where(vis > 0, x_c, torch.full_like(x_c, +10.0)).amin(dim=-1, keepdim=True) * any_vis
            H_box = torch.clamp(y_max - y_min, min=0.0).squeeze(-1)
            W_box = torch.clamp(x_max - x_min, min=0.0).squeeze(-1)
            aspect_ar = (H_box - W_box) / (H_box + W_box + 1e-6)
            h_hip = -ground_y.squeeze(-1)
            vy_hip = torch.zeros_like(h_hip); vy_hip[:, 1:] = h_hip[:, 1:] - h_hip[:, :-1]
            ay_hip = torch.zeros_like(h_hip); ay_hip[:, 2:] = vy_hip[:, 2:] - vy_hip[:, 1:-1]
            grav_sig = torch.stack([cos_theta, abs_sin_theta, aspect_ar, h_hip, vy_hip, ay_hip], dim=1)
            head_extras.append(self.grav_mlp(self._summarize_temporal_signals(grav_sig)))

        if self.use_wrist_head_feats:
            nose_x, nose_y, nose_vis = x_c[:, :, 0:1], y_c[:, :, 0:1], vis[:, :, 0:1]
            dx_nose = (x_c - nose_x) * vis * nose_vis
            dy_nose = (y_c - nose_y) * vis * nose_vis
            node_extras.append(torch.stack([dx_nose, dy_nose], dim=1))
            d_lw = torch.sqrt((x_c[:, :, 9] - x_c[:, :, 0]) ** 2 + (y_c[:, :, 9] - y_c[:, :, 0]) ** 2 + 1e-6)
            d_rw = torch.sqrt((x_c[:, :, 10] - x_c[:, :, 0]) ** 2 + (y_c[:, :, 10] - y_c[:, :, 0]) ** 2 + 1e-6)
            d_min = torch.minimum(d_lw, d_rw)
            dy_w_max = torch.maximum(y_c[:, :, 9] - y_c[:, :, 0], y_c[:, :, 10] - y_c[:, :, 0])
            p_xy = raw_x[:, :2, :, :]
            cos_el_l = self._joint_angle_cos(p_xy[:, :, :, 5], p_xy[:, :, :, 7], p_xy[:, :, :, 9]).squeeze(1)
            cos_el_r = self._joint_angle_cos(p_xy[:, :, :, 6], p_xy[:, :, :, 8], p_xy[:, :, :, 10]).squeeze(1)
            wrist_sig = torch.stack([d_lw, d_rw, d_min, dy_w_max, cos_el_l, cos_el_r], dim=1)
            head_extras.append(self.wrist_mlp(self._summarize_temporal_signals(wrist_sig)))

        if self.use_lower_body_feats:
            ank_y = 0.5 * (y_c[:, :, 15:16] + y_c[:, :, 16:17])
            dy_ank = (y_c - ank_y) * vis
            vy_ank = torch.zeros_like(dy_ank)
            vy_ank[:, 1:, :] = (dy_ank[:, 1:, :] - dy_ank[:, :-1, :]) * vis[:, 1:, :] * vis[:, :-1, :]
            node_extras.append(torch.stack([dy_ank, vy_ank], dim=1))
            p_xy = raw_x[:, :2, :, :]
            cos_kn_l = self._joint_angle_cos(p_xy[:, :, :, 11], p_xy[:, :, :, 13], p_xy[:, :, :, 15]).squeeze(1)
            cos_kn_r = self._joint_angle_cos(p_xy[:, :, :, 12], p_xy[:, :, :, 14], p_xy[:, :, :, 16]).squeeze(1)
            kn_y_mid = 0.5 * (y_c[:, :, 13] + y_c[:, :, 14])
            ank_y_mid = 0.5 * (y_c[:, :, 15] + y_c[:, :, 16])
            lower_sig = torch.stack([cos_kn_l, cos_kn_r, -kn_y_mid, kn_y_mid - ank_y_mid, -ank_y_mid, (x_c[:, :, 15] - x_c[:, :, 16]).abs()], dim=1)
            head_extras.append(self.lower_mlp(self._summarize_temporal_signals(lower_sig)))

        return node_extras, head_extras

    def _compute_kinematics(self, x: torch.Tensor) -> torch.Tensor:
        x_parent = x[:, :, :, self.parents]
        both_vis = ((x[:, 2:3, :, :] > 0) & (x_parent[:, 2:3, :, :] > 0)).float()
        bone_xy  = (x[:, :2, :, :] - x_parent[:, :2, :, :]) * both_vis
        bone_len = torch.sqrt((bone_xy ** 2).sum(dim=1, keepdim=True) + 1e-8) * both_vis
        bone     = torch.cat([bone_xy, bone_len], dim=1)

        vel_xy = torch.zeros_like(x[:, :2, :, :])
        both_t_vis = ((x[:, 2:3, 1:, :] > 0) & (x[:, 2:3, :-1, :] > 0)).float()
        vel_xy[:, :, 1:, :] = (x[:, :2, 1:, :] - x[:, :2, :-1, :]) * both_t_vis
        vel_mag = torch.sqrt((vel_xy ** 2).sum(dim=1, keepdim=True) + 1e-8)
        motion  = torch.cat([vel_xy, vel_mag], dim=1)

        return torch.cat([x, bone, motion], dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raw_x = x
        node_extras, head_extras = self._compute_v31_branches(raw_x)
        if self.use_kinematic_feats:
            x = self._compute_kinematics(raw_x)
        if node_extras:
            x = torch.cat([x] + node_extras, dim=1)
        B, C, T, V = x.shape
        x_bn = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x_bn = self.data_bn(x_bn).view(B, C, V, T).permute(0, 1, 3, 2)

        h = self.b1(x_bn)
        h = self.b2(h)
        h = self.b3(h)
        h = self.b4(h)

        h_v = h.mean(dim=-1)
        if self.use_temporal_pyr:
            h_perm = h_v.permute(0, 2, 1)
            att_w  = torch.softmax(self.temp_att(h_perm), dim=1)
            h_att  = (h_perm * att_w).sum(dim=1)
            mid    = max(1, h_v.shape[2] // 2)
            h_diff = h_v[:, :, mid:].mean(dim=2) - h_v[:, :, :mid].mean(dim=2)
            h_max  = h.amax(dim=(2, 3))
            feat   = torch.cat([h_att, h_diff, h_max], dim=1)
        else:
            feat = h_v.mean(dim=-1)
        if head_extras:
            feat = torch.cat([feat] + head_extras, dim=1)
        return self.head(feat)

## 3. Load Leak-Free Video-Level Split Dataset

In [ ]:
DATA_DIR = Path('data/skeleton_dataset/windowed')
if not (DATA_DIR / 'X_train.npy').exists() and Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').glob('**/X_train.npy'))
    if candidates:
        DATA_DIR = candidates[0].parent

print(f'Loading leak-free splits from: {DATA_DIR}')
X_train_raw = np.load(DATA_DIR / 'X_train.npy')
y_train     = np.load(DATA_DIR / 'y_train.npy')
X_val_raw   = np.load(DATA_DIR / 'X_val.npy')
y_val       = np.load(DATA_DIR / 'y_val.npy')
X_test_raw  = np.load(DATA_DIR / 'X_test.npy')
y_test      = np.load(DATA_DIR / 'y_test.npy')

cids_val  = np.load(DATA_DIR / 'clip_ids_val.npy')  if (DATA_DIR / 'clip_ids_val.npy').exists()  else None
cids_test = np.load(DATA_DIR / 'clip_ids_test.npy') if (DATA_DIR / 'clip_ids_test.npy').exists() else None
val_seq_path  = DATA_DIR / 'val_sequences.npz'
test_seq_path = DATA_DIR / 'test_sequences.npz'

CLASS_NAMES = ['Walk', 'Eat', 'Drink', 'Fall', 'Lying', 'Sit_Down', 'Stand_Up', 'Crouching']
NUM_CLASSES = len(CLASS_NAMES)
LABEL_TO_ALERT = {
    'Walk': 'GREEN', 'Eat': 'GREEN', 'Drink': 'GREEN', 'Sit_Down': 'GREEN', 'Stand_Up': 'GREEN',
    'Crouching': 'YELLOW', 'Lying': 'YELLOW',
    'Fall': 'RED',
}

X_train = X_train_raw.transpose(0, 3, 1, 2).astype(np.float32)
X_val   = X_val_raw.transpose(0, 3, 1, 2).astype(np.float32)
X_test  = X_test_raw.transpose(0, 3, 1, 2).astype(np.float32)

classes, counts = np.unique(y_train, return_counts=True)
class_weights = len(y_train) / (NUM_CLASSES * counts.astype(np.float32))
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f'Shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}')
print(f'Train Post-Augmentation Imbalance Ratio: {counts.max()}/{counts.min()} = {counts.max()/counts.min():.2f}x')

## 4. Final Controlled Ablation on Validation Set (`v3.0-clean` vs `v3.1-B` vs `v3.1-C` vs `v3.1-D4 [B+C]`)
Evaluated **strictly on Validation (`X_val`: `830` windows / `143` clips)** under identical seed and training protocol:
- `v3.0-clean`: Baseline `v2.4` architecture on Dataset V3b (`279,251` params)
- `v3.1-B`: `v3.0-clean` + Wrist-to-Head Features (`280,999` params)
- `v3.1-C`: `v3.0-clean` + Lower-Body Kinematics (`280,999` params)
- `v3.1-D4`: `v3.0-clean` + **Wrist-to-Head + Lower-Body Fusion (`B + C`, `282,747` params)**

In [ ]:
BATCH_SIZE = 64
EPOCHS = 60
LR = 1.5e-3

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).long()),
    batch_size=BATCH_SIZE, shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val).long()),
    batch_size=BATCH_SIZE, shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).long()),
    batch_size=BATCH_SIZE, shuffle=False,
)

def compute_safety_metrics(y_true, y_pred, fall_idx=3):
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    fall_mask = (y_true == fall_idx)
    pred_fall = (y_pred == fall_idx)
    tp = int((fall_mask & pred_fall).sum())
    fp = int((~fall_mask & pred_fall).sum())
    fn = int((fall_mask & ~pred_fall).sum())
    fall_rec  = tp / max(1, tp + fn)
    fall_prec = tp / max(1, tp + fp)
    fall_f1   = 2 * fall_rec * fall_prec / max(1e-8, fall_rec + fall_prec)
    safety_score = 0.50 * fall_rec + 0.25 * fall_f1 + 0.25 * macro_f1
    return {
        'safety_score': safety_score,
        'fall_class_recall': fall_rec * 100.0,
        'fall_class_precision': fall_prec * 100.0,
        'fall_class_f1': fall_f1 * 100.0,
        'macro_f1': macro_f1,
        'acc': float((y_true == y_pred).mean() * 100.0),
    }

ABLATION_CONFIGS = [
    ('ST-GCN v3.0-clean (Baseline v2.4 Arch)',                    dict(use_temporal_pyr=True, use_adaptive_adj=True, use_kinematic_feats=True, use_gravity_feats=False, use_wrist_head_feats=False, use_lower_body_feats=False)),
    ('ST-GCN v3.1-B (+ Wrist-to-Head Features)',                  dict(use_temporal_pyr=True, use_adaptive_adj=True, use_kinematic_feats=True, use_gravity_feats=False, use_wrist_head_feats=True,  use_lower_body_feats=False)),
    ('ST-GCN v3.1-C (+ Lower-Body Kinematics)',                   dict(use_temporal_pyr=True, use_adaptive_adj=True, use_kinematic_feats=True, use_gravity_feats=False, use_wrist_head_feats=False, use_lower_body_feats=True)),
    ('ST-GCN v3.1-D4 (+ Wrist-to-Head + Lower-Body Fusion B+C)',  dict(use_temporal_pyr=True, use_adaptive_adj=True, use_kinematic_feats=True, use_gravity_feats=False, use_wrist_head_feats=True,  use_lower_body_feats=True)),
]

ablation_results = []
overall_best_score = -1.0
overall_best_weights = None
overall_best_cfg = None
overall_best_name = ''

for var_name, arch_cfg in ABLATION_CONFIGS:
    for use_w in (False, True):
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        m = ActionSTGCN(in_channels=3, num_classes=NUM_CLASSES, num_joints=17, **arch_cfg).to(device)
        crit = nn.CrossEntropyLoss(weight=weights_tensor if use_w else None)
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        best_sc = -1.0
        best_st = None
        best_m = {}
        loss_tag = 'Weighted' if use_w else 'Unweighted'

        for epoch in range(1, EPOCHS + 1):
            m.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = crit(m(xb), yb)
                loss.backward()
                opt.step()
            sched.step()

            m.eval()
            v_preds, v_targets = [], []
            with torch.no_grad():
                for xb, yb in val_loader:
                    v_preds.extend(m(xb.to(device)).argmax(1).cpu().numpy())
                    v_targets.extend(yb.numpy())
            metrics = compute_safety_metrics(np.array(v_targets), np.array(v_preds))
            metrics['epoch'] = epoch
            if metrics['safety_score'] > best_sc:
                best_sc = metrics['safety_score']
                best_st = {k: v.cpu().clone() for k, v in m.state_dict().items()}
                best_m = metrics

        full_label = f'{var_name} [{loss_tag}]'
        print(f'{full_label:<58} | ValSafety={best_m["safety_score"]:.4f} | ValFallRec={best_m["fall_class_recall"]:5.1f}% | ValFallPrec={best_m["fall_class_precision"]:5.1f}% | ValMacroF1={best_m["macro_f1"]:.4f} | ValAcc={best_m["acc"]:5.1f}% (ep {best_m["epoch"]})')
        ablation_results.append((full_label, best_m))
        if best_sc > overall_best_score:
            overall_best_score = best_sc
            overall_best_weights = best_st
            overall_best_cfg = arch_cfg
            overall_best_name = full_label

print('\n' + '=' * 90)
print(f'SELECTED WINNER ON VALIDATION: {overall_best_name} (Val Safety Score = {overall_best_score:.4f})')
print('=' * 90)

model = ActionSTGCN(in_channels=3, num_classes=NUM_CLASSES, num_joints=17, **overall_best_cfg).to(device)
model.load_state_dict(overall_best_weights)
model.eval()

# 4B. Validation Threshold Grid Search (CRITICAL_THRESHOLD x INSTANT_RED_THRESHOLD)
BEST_CRIT_TH, BEST_INST_TH = 0.75, 0.90
if val_seq_path.exists():
    val_seqs = np.load(val_seq_path)
    def precompute_val_traces(replay_fps: float):
        traces = []
        with torch.no_grad():
            for clip_key in sorted(val_seqs.files):
                cls_name = clip_key.split('/')[0]
                true_is_fall = (cls_name == 'Fall')
                seq_10fps = val_seqs[clip_key]
                duration = len(seq_10fps) / 10.0
                target_times = np.arange(0.0, duration, 1.0 / replay_fps, dtype=np.float64)
                idx_arr = np.clip(np.floor(target_times * 10.0 + 0.5).astype(int), 0, max(0, len(seq_10fps) - 1))
                seq = seq_10fps[idx_arr]
                frame_times = idx_arr.astype(np.float64) * 0.10
                if len(seq) < 16:
                    pad_n = 16 - len(seq)
                    seq = np.concatenate([seq, np.tile(seq[-1:], (pad_n, 1, 1))], axis=0)
                    pad_times = frame_times[-1] + np.arange(1, pad_n + 1, dtype=np.float64) / replay_fps
                    frame_times = np.concatenate([frame_times, pad_times], axis=0)
                buf, vf = collections.deque(maxlen=16), collections.deque(maxlen=16)
                miss = 0
                ticks = []
                for kp, t_sec in zip(seq, frame_times):
                    if bool(np.all(kp == 0.0)):
                        miss += 1
                        if miss > 3:
                            buf.clear(); vf.clear(); ticks.append(('FLUSH', 0.0, float(t_sec))); continue
                        buf.append(np.zeros((17, 3), dtype=np.float32)); vf.append(False)
                    else:
                        miss = 0; buf.append(kp.astype(np.float32)); vf.append(True)
                    if len(buf) < 16 or sum(vf) < 8:
                        if len(buf) == 16 and sum(vf) < 8:
                            ticks.append(('LOW_SUBJ', 0.0, float(t_sec)))
                        continue
                    win = np.stack(list(buf), axis=0)
                    win_t = torch.from_numpy(win.transpose(2, 0, 1)[np.newaxis, ...].astype(np.float32)).to(device)
                    pr = torch.softmax(model(win_t), dim=-1)[0].cpu().numpy()
                    idx = int(np.argmax(pr))
                    ticks.append((CLASS_NAMES[idx], float(pr[idx]), float(t_sec)))
                traces.append((clip_key, true_is_fall, ticks))
        return traces

    def eval_traces_grid(traces, crit_th: float, inst_th: float):
        tp, fp, nf, nnf, n_inst, n_3 = 0, 0, 0, 0, 0, 0
        delays, missed, fp_clips = [], [], []
        for key, is_fall, ticks in traces:
            cc, fc, red = 0, None, False
            for lbl, conf, t_sec in ticks:
                if lbl in ('FLUSH', 'LOW_SUBJ') or conf < 0.50:
                    cc = 0; fc = None; continue
                if lbl == 'Fall':
                    if conf >= inst_th:
                        if not red:
                            red = True
                            if is_fall:
                                n_inst += 1; delays.append(0.0 if fc is None else t_sec - fc)
                        cc = 0
                    elif conf >= crit_th:
                        if cc == 0: fc = t_sec
                        cc += 1
                        if cc >= 3:
                            if not red:
                                red = True
                                if is_fall and fc is not None:
                                    n_3 += 1; delays.append(t_sec - fc)
                            cc = 0
                    else:
                        cc = 0; fc = None
                else:
                    cc = 0; fc = None
            if is_fall:
                nf += 1
                if red: tp += 1
                else: missed.append(key.split('/')[-1])
            else:
                nnf += 1
                if red: fp += 1; fp_clips.append(key)
        return dict(tp=tp, fp=fp, nf=nf, nnf=nnf, n_inst=n_inst, n_3=n_3, mean_delay=float(np.mean(delays)) if delays else 0.0, missed=missed, fp_clips=fp_clips)

    traces_10 = precompute_val_traces(10.0)
    traces_83 = precompute_val_traces(8.33)
    print('\n' + '=' * 108)
    print('4B. VALIDATION THRESHOLD GRID SEARCH ON WINNING MODEL (10.0 FPS & 8.33 FPS, N_CRIT=3, N_REC=5):')
    print('=' * 108)
    print('Crit | Inst | 10FPS_Val_RED_Rec | 10FPS_Val_RED_FPR | 8.3FPS_Val_RED_Rec | 8.3FPS_Val_RED_FPR | 10FPS_Inst/3T | Delay10 | Missed10')
    print('-' * 108)
    best_val_util = -1e9
    for c_th in [0.55, 0.60, 0.65, 0.70, 0.75]:
        for i_th in [0.80, 0.85, 0.90, 0.95]:
            r10 = eval_traces_grid(traces_10, c_th, i_th)
            r83 = eval_traces_grid(traces_83, c_th, i_th)
            rec10 = r10['tp'] / max(1, r10['nf'])
            fpr10 = r10['fp'] / max(1, r10['nnf'])
            rec83 = r83['tp'] / max(1, r83['nf'])
            fpr83 = r83['fp'] / max(1, r83['nnf'])
            util = (0.6 * rec10 + 0.4 * rec83) - 1.5 * (0.6 * fpr10 + 0.4 * fpr83) + 0.01 * c_th
            if util > best_val_util:
                best_val_util = util
                BEST_CRIT_TH, BEST_INST_TH = c_th, i_th
            print(f'{c_th:4.2f} | {i_th:4.2f} | {r10["tp"]:2d}/{r10["nf"]:2d} ({100*rec10:6.2f}%) | {r10["fp"]:2d}/{r10["nnf"]:3d} ({100*fpr10:6.2f}%) | {r83["tp"]:2d}/{r83["nf"]:2d} ({100*rec83:6.2f}%)  | {r83["fp"]:2d}/{r83["nnf"]:3d} ({100*fpr83:6.2f}%)  |   {r10["n_inst"]:2d}/{r10["n_3"]:<2d}       | {r10["mean_delay"]:6.3f}s | {r10["missed"]}')
    r_best10 = eval_traces_grid(traces_10, BEST_CRIT_TH, BEST_INST_TH)
    print('=' * 108)
    print(f'SELECTED VALIDATION THRESHOLDS: CRITICAL_THRESHOLD={BEST_CRIT_TH:.2f}, INSTANT_RED_THRESHOLD={BEST_INST_TH:.2f}')
    print(f'Validation False-Positive Non-Fall Clips @ Selected Thresholds (10 FPS): {r_best10["fp_clips"]}')
    print('=' * 108)

## 5. Held-Out Test Set Evaluation (Run Once on Validation Winner)

In [ ]:
model.eval()
all_preds, all_probs, all_targets = [], [], []

latencies = []
sample_input = torch.from_numpy(X_test[:1]).to(device)
with torch.no_grad():
    for _ in range(10): _ = model(sample_input)
    for _ in range(100):
        t0 = time.perf_counter()
        _ = model(sample_input)
        latencies.append((time.perf_counter() - t0) * 1000.0)
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_targets.extend(yb.numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
fall_idx = CLASS_NAMES.index('Fall')

print(f'Mean inference latency: {np.mean(latencies):.3f} ms +/- {np.std(latencies):.3f} ms per sequence')
print('\n' + '=' * 76)
print(f'1. WINDOW-LEVEL Classification Report ({len(all_targets)} Windows, stride=4 @ exact 10.0 FPS):')
print('=' * 76)
print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, digits=4))
print('8x8 Window-Level Confusion Matrix (rows=true, cols=pred):')
print(confusion_matrix(all_targets, all_preds))

if cids_test is not None and len(cids_test) == len(all_targets):
    v1_mask = np.array([not str(c).split('/')[-1].startswith('v2_') for c in cids_test])
    v2_mask = ~v1_mask
    if 0 < int(v1_mask.sum()) < len(all_targets):
        print('\n' + '=' * 76)
        print(f'1B. V1_LEGACY_TEST — Legacy Holdout Test ({int(v1_mask.sum())} Windows / 83 Clips):')
        print('=' * 76)
        print(classification_report(all_targets[v1_mask], all_preds[v1_mask], target_names=CLASS_NAMES, digits=4))
        print('\n' + '=' * 76)
        print(f'1C. V2_GROUP_HELDOUT_TEST — Group-Heldout HD Test ({int(v2_mask.sum())} Windows / 64 Clips):')
        print('=' * 76)
        print(classification_report(all_targets[v2_mask], all_preds[v2_mask], target_names=CLASS_NAMES, digits=4, zero_division=0))
    unique_clips = sorted(set(cids_test.tolist()))
    clip_true, clip_pred_mean = [], []
    for cid in unique_clips:
        mask = (cids_test == cid)
        clip_true.append(int(all_targets[mask][0]))
        clip_pred_mean.append(int(np.argmax(all_probs[mask].mean(axis=0))))
    print('\n' + '=' * 76)
    print(f'2A. VIDEO-LEVEL Classification Report — Mean Prob ({len(unique_clips)} Independent Clips):')
    print('=' * 76)
    print(classification_report(np.array(clip_true), np.array(clip_pred_mean), target_names=CLASS_NAMES, digits=4))

if test_seq_path.exists():
    test_seqs = np.load(test_seq_path)
    def run_stream_replay(replay_fps: float, subset: str = 'ALL', crit_th: float = 0.75, inst_th: float = 0.90):
        tp_r1, fp_r1, tp_r2, fp_r2 = 0, 0, 0, 0
        n_fall_vids, n_nonfall_vids = 0, 0
        n_instant, n_three_tick = 0, 0
        trigger_delays_sec = []
        with torch.no_grad():
            for clip_key in sorted(test_seqs.files):
                cls_name, stem = clip_key.split('/')[0], clip_key.split('/')[-1]
                is_v2 = stem.startswith('v2_')
                if subset == 'V1_LEGACY_TEST' and is_v2: continue
                if subset == 'V2_GROUP_HELDOUT_TEST' and not is_v2: continue
                true_is_fall = (cls_name == 'Fall')
                seq_10fps = test_seqs[clip_key]
                duration = len(seq_10fps) / 10.0
                target_times = np.arange(0.0, duration, 1.0 / replay_fps, dtype=np.float64)
                idx_arr = np.clip(np.floor(target_times * 10.0 + 0.5).astype(int), 0, max(0, len(seq_10fps) - 1))
                seq = seq_10fps[idx_arr]
                frame_times = idx_arr.astype(np.float64) * 0.10
                if len(seq) < 16:
                    pad_n = 16 - len(seq)
                    seq = np.concatenate([seq, np.tile(seq[-1:], (pad_n, 1, 1))], axis=0)
                    pad_times = frame_times[-1] + np.arange(1, pad_n + 1, dtype=np.float64) / replay_fps
                    frame_times = np.concatenate([frame_times, pad_times], axis=0)
                buf = collections.deque(maxlen=16)
                valid_flags = collections.deque(maxlen=16)
                consec_missing = 0
                crit_count = 0
                red_triggered = False
                any_fall_tick = False
                first_crit_time = None
                for frame_kp, t_sec in zip(seq, frame_times):
                    if bool(np.all(frame_kp == 0.0)):
                        consec_missing += 1
                        if consec_missing > 3:
                            buf.clear(); valid_flags.clear(); crit_count = 0; first_crit_time = None
                            continue
                        buf.append(np.zeros((17, 3), dtype=np.float32)); valid_flags.append(False)
                    else:
                        consec_missing = 0
                        buf.append(frame_kp.astype(np.float32)); valid_flags.append(True)
                    if len(buf) < 16 or sum(valid_flags) < 8:
                        if len(buf) == 16 and sum(valid_flags) < 8:
                            crit_count = 0; first_crit_time = None
                        continue
                    win = np.stack(list(buf), axis=0)
                    win_t = torch.from_numpy(win.transpose(2, 0, 1)[np.newaxis, ...].astype(np.float32)).to(device)
                    probs_t = torch.softmax(model(win_t), dim=-1)[0].cpu().numpy()
                    pred_idx = int(np.argmax(probs_t))
                    pred_conf = float(probs_t[pred_idx])
                    pred_label = CLASS_NAMES[pred_idx]
                    if pred_conf < 0.50:
                        crit_count = 0; first_crit_time = None; continue
                    if pred_label == 'Fall':
                        any_fall_tick = True
                        if pred_conf >= inst_th:
                            if not red_triggered:
                                red_triggered = True
                                if true_is_fall:
                                    n_instant += 1
                                    trigger_delays_sec.append(0.0 if first_crit_time is None else float(t_sec - first_crit_time))
                            crit_count = 0
                        elif pred_conf >= crit_th:
                            if crit_count == 0: first_crit_time = float(t_sec)
                            crit_count += 1
                            if crit_count >= 3:
                                if not red_triggered:
                                    red_triggered = True
                                    if true_is_fall and first_crit_time is not None:
                                        n_three_tick += 1
                                        trigger_delays_sec.append(float(t_sec - first_crit_time))
                                crit_count = 0
                        else:
                            crit_count = 0; first_crit_time = None
                    else:
                        crit_count = 0; first_crit_time = None
                if true_is_fall:
                    n_fall_vids += 1
                    if any_fall_tick: tp_r1 += 1
                    if red_triggered: tp_r2 += 1
                else:
                    n_nonfall_vids += 1
                    if any_fall_tick: fp_r1 += 1
                    if red_triggered: fp_r2 += 1
        d_arr = np.array(trigger_delays_sec, dtype=np.float64) if trigger_delays_sec else np.zeros((1,), dtype=np.float64)
        return {'n_fall': n_fall_vids, 'n_nonfall': n_nonfall_vids, 'tp_r1': tp_r1, 'fp_r1': fp_r1, 'tp_r2': tp_r2, 'fp_r2': fp_r2, 'n_instant': n_instant, 'n_three_tick': n_three_tick, 'mean_delay_sec': float(np.mean(d_arr)), 'med_delay_sec': float(np.median(d_arr)), 'max_delay_sec': float(np.max(d_arr))}

    th_configs = [('Default Thresholds (Crit=0.75, Inst=0.90)', 0.75, 0.90)]
    if (BEST_CRIT_TH, BEST_INST_TH) != (0.75, 0.90):
        th_configs.append((f'Val-Tuned Thresholds (Crit={BEST_CRIT_TH:.2f}, Inst={BEST_INST_TH:.2f})', BEST_CRIT_TH, BEST_INST_TH))
    for th_label, c_th, i_th in th_configs:
        print('\n' + '=' * 84)
        print(f'2B. STREAM-CADENCE VIDEO-LEVEL TRIGGER EVALUATION — {th_label}:')
        print('=' * 84)
        for mode_label, fps_val in [('Target-Cadence Replay (10.0 FPS design)', 10.0), ('CPU-Realistic Replay (8.33 FPS measured)', 8.33)]:
            print(f'\n  >>> {mode_label} <<<')
            for sub_tag in ['V1_LEGACY_TEST', 'V2_GROUP_HELDOUT_TEST', 'ALL']:
                res = run_stream_replay(fps_val, subset=sub_tag, crit_th=c_th, inst_th=i_th)
                nf, nnf = res['n_fall'], res['n_nonfall']
                print(f'  [{sub_tag:<21}] Fall Recall (Rule1): {res["tp_r1"]:2d}/{nf:2d} ({100*res["tp_r1"]/max(1,nf):5.2f}%) | RED Trigger Recall: {res["tp_r2"]:2d}/{nf:2d} ({100*res["tp_r2"]/max(1,nf):5.2f}%) | RED FPR: {res["fp_r2"]:2d}/{nnf:3d} ({100*res["fp_r2"]/max(1,nnf):5.2f}%)')
                print(f'                          Instant RED: {res["n_instant"]} | 3-Tick RED: {res["n_three_tick"]} | Delay mean={res["mean_delay_sec"]:.3f}s, med={res["med_delay_sec"]:.3f}s, max={res["max_delay_sec"]:.3f}s')

## 6. Export Winning ST-GCN v3.1 Checkpoint

In [ ]:
out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('models')
out_dir.mkdir(exist_ok=True)
save_path = out_dir / 'action_stgcn.pth'
torch.save(overall_best_weights, str(save_path))
print(f'Successfully saved winning ST-GCN v3.1 weights ({overall_best_name}) to: {save_path}')